In [1]:
import os

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score, classification_report
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import lightgbm as lgb

from eyf.clase_ternaria import calcular_clase_ternaria
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Preparación

## Datos

In [2]:
# Definir rutas
DATA_PATH = '../data'
RAW_DATA_NAME = 'competencia_01_crudo.csv'
RAW_DATA_PATH = f'{DATA_PATH}/{RAW_DATA_NAME}'
TARGET_DATA_NAME = 'competencia_01_target.csv'
TARGET_DATA_PATH = f'{DATA_PATH}/{TARGET_DATA_NAME}'

In [3]:
if not os.path.exists(TARGET_DATA_PATH):
    print("No existe el archivo de target, calculando...")
    # Cargar los datos
    print("Cargando datos...")
    df = pd.read_csv(RAW_DATA_PATH)
    print(f"Datos cargados: {df.shape[0]} filas y {df.shape[1]} columnas")

    # Calcular la clase ternaria (variable objetivo)
    print("Calculando clase ternaria...")
    df = calcular_clase_ternaria(df)

    # Verificar la distribución de clases
    print("\nDistribución de la clase ternaria:")
    print(df['clase_ternaria'].value_counts())
    print("\nPorcentajes:")
    print(df['clase_ternaria'].value_counts(normalize=True) * 100)
    df.to_csv(f'{DATA_PATH}/competencia_01_target.csv', index=False)
else:
    print("Existe el archivo de target, cargando...")
    df = pd.read_csv(TARGET_DATA_PATH)

Existe el archivo de target, cargando...


In [4]:
df['clase_peso'] = 1.0

df.loc[df['clase_ternaria'] == 'BAJA+2', 'clase_peso'] = 1.00002
df.loc[df['clase_ternaria'] == 'BAJA+1', 'clase_peso'] = 1.00001

In [ ]:
def prepare_data(df):

    exclude_cols = ['numero_de_cliente', 'foto_mes', 'clase_ternaria', 'target_binario', 'clase_peso']
    # Calcular lag 1 para todas las variables que no están en exclude_cols
    lag_cols = [col for col in df.columns if col not in exclude_cols]
    for col in lag_cols:
        df[f'{col}_lag1'] = df.groupby('numero_de_cliente')[col].shift(1)
    bank_cat_cols = ["active_quarter", "cliente_vip", "internet",
    "cdescubierto_preacordado", "ccaja_seguridad", "tcallcenter",
    "thomebanking", "tmobile_app", 
    ]
    master_cat_cols = ["Master_delinquency", "Master_status"]
    visa_cat_cols = ["Visa_delinquency", "Visa_status"]
    visa_num_cols = df.columns[df.columns.str.contains('Visa')]
    visa_num_cols = [col for col in visa_num_cols if col not in visa_cat_cols]
    master_num_cols = df.columns[df.columns.str.contains('Master')]
    master_num_cols = [col for col in master_num_cols if col not in master_cat_cols]
    bank_num_cols = [col for col in df.columns if col not in bank_cat_cols + master_cat_cols + visa_cat_cols + visa_num_cols + exclude_cols]

    train_data = df[df['foto_mes'].isin([202102, 202103])].copy()
    test_data = df[df['foto_mes'] == 202104].copy()
    w_test = test_data['clase_peso']
    w_train = train_data['clase_peso']
    
    feature_cols = [col for col in train_data.columns if col not in exclude_cols]

    X_train = train_data[feature_cols]
    X_test = test_data[feature_cols]

    X_cat_train = X_train[bank_cat_cols + master_cat_cols + visa_cat_cols]
    X_cat_test = X_test[bank_cat_cols + master_cat_cols + visa_cat_cols]
    X_num_train = X_train[bank_num_cols + master_num_cols + visa_num_cols]
    X_num_test = X_test[bank_num_cols + master_num_cols + visa_num_cols]

    y_train = train_data['clase_binaria']
    y_test = test_data['clase_binaria']

    scaler = StandardScaler()
    X_num_train = scaler.fit_transform(X_num_train)
    X_num_test = scaler.transform(X_num_test)

    encoder = OneHotEncoder(sparse_output=False)
    # TODO: Esto se ve feo
    X_cat_total = pd.concat([X_cat_train, X_cat_test], axis=0)
    encoder.fit(X_cat_total)
    X_cat_train = encoder.transform(X_cat_train)
    X_cat_test = encoder.transform(X_cat_test)

    return X_num_train, X_cat_train, X_num_test, X_cat_test, y_train.values, y_test.values, w_train, w_test, scaler, encoder


In [6]:
X_num_train, X_cat_train, X_num_test, X_cat_test, y_train, y_test,w_train, w_test, scaler, encoder = prepare_data(df)

In [7]:
X_train = np.hstack([X_num_train, X_cat_train])
X_test = np.hstack([X_num_test, X_cat_test])

# Llenar los valores NaN con la mediana de cada columna en los conjuntos de entrenamiento y prueba
X_train = pd.DataFrame(X_train)
X_test = pd.DataFrame(X_test)

X_train = X_train.fillna(X_train.median())
X_test = X_test.fillna(X_train.median())  # Usar la mediana de entrenamiento para evitar data leakage

X_train = X_train.values
X_test = X_test.values




## Parámetros

In [ ]:
# Hyperparámetros del autoencoder
INPUT_DIM = X_train.shape[1]
LAYERS = [128,64,32,16]    # dimensión del embedding latente
BATCH_SIZE = 128
LR = 1e-3
NUM_EPOCHS = 50

class TabularDataset(Dataset):
    def __init__(self, X: np.ndarray):
        self.X = X.astype(np.float32)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.X[idx]  # para autoencoder, input = target

# --- Autoencoder configurable (fully connected) ---
class Autoencoder(nn.Module):
    def __init__(self, input_dim, encoder_layer_sizes):
        """
        input_dim: dimensión de entrada
        encoder_layer_sizes: lista con el tamaño de cada capa del encoder, 
                             el último elemento es la dimensión del embedding latente
        """
        super().__init__()
        # Construir encoder
        encoder_layers = []
        prev_dim = input_dim
        for layer_dim in encoder_layer_sizes:
            encoder_layers.append(nn.Linear(prev_dim, layer_dim))
            encoder_layers.append(nn.ReLU())
            prev_dim = layer_dim
        encoder_layers = encoder_layers[:-1]  # quitar la última ReLU para el embedding
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Construir decoder (simétrico)
        decoder_layer_sizes = list(reversed(encoder_layer_sizes[:-1])) + [input_dim]
        decoder_layers = []
        prev_dim = encoder_layer_sizes[-1]
        for layer_dim in decoder_layer_sizes:
            decoder_layers.append(nn.Linear(prev_dim, layer_dim))
            decoder_layers.append(nn.ReLU())
            prev_dim = layer_dim
        decoder_layers = decoder_layers[:-1]  # quitar la última ReLU para la salida
        self.decoder = nn.Sequential(*decoder_layers)
        
    def forward(self, x):
        z = self.encoder(x)
        x_rec = self.decoder(z)
        return x_rec
    def encode(self, x):
        return self.encoder(x)



In [26]:
# --- Entrenamiento del autoencoder ---
def train_autoencoder(model, dataloader, num_epochs=50, lr=1e-3, device="cpu"):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for xb, yb in dataloader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            xb_rec = model(xb)
            loss = criterion(xb_rec, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
        epoch_loss = running_loss / len(dataloader.dataset)
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, loss = {epoch_loss:.6f}")
    return model

# --- Extraer embeddings (codificación) ---
def get_embeddings(model, X: np.ndarray, device="cpu", batch_size=128):
    model = model.to(device)
    model.eval()
    ds = TabularDataset(X)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)
    zs = []
    with torch.no_grad():
        for xb, _ in dl:
            xb = xb.to(device)
            z = model.encode(xb)
            zs.append(z.cpu().numpy())
    Z = np.vstack(zs)
    return Z  # shape (n_samples, embed_dim)

# --- Entrenar LightGBM sobre embeddings + (opcionalmente) features originales ---
ganancia_acierto = 780000
costo_estimulo = 20000

def lgb_gan_eval(y_pred, data):
    weight = data.get_weight()
    ganancia = np.where(weight == 1.00002, ganancia_acierto, 0) - np.where(weight < 1.00002, costo_estimulo, 0)
    ganancia = ganancia[np.argsort(y_pred)[::-1]]
    ganancia = np.cumsum(ganancia)

    return 'gan_eval', np.max(ganancia) , True
def train_lgbm(Z_train, y_train, Z_test, y_test, w_train, w_test):


    # Configuración simple/por defecto para LightGBM binario
    params = {
        'objective': 'binary',
        'metric': 'gan_eval',
        'boosting_type': 'gbdt',
        'max_bin': 31,
        'num_leaves': 31,
        'learning_rate': 0.001,
        'feature_fraction': 0.3,
        'bagging_fraction': 0.7,
        'verbose': 0
    }
    train_dataset = lgb.Dataset(Z_train, label=y_train, weight=w_train)
    test_dataset = lgb.Dataset(Z_test, label=y_test, reference=train_dataset, weight=w_test)
    model = lgb.train(
    params,
    train_dataset,
    num_boost_round=10000,
    valid_sets=[train_dataset, test_dataset],
    valid_names=['train', 'test'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=200),
        lgb.log_evaluation(period=100)
    ],
    feval=lgb_gan_eval,
)

    print(f"✅ Modelo entrenado exitosamente!")
    print(f"Mejor iteración: {model.best_iteration}")
    print(f"Mejor ganancia de validación: {model.best_score['test']['gan_eval']:.6f}")
    print(f"Mejor ganancia de training: {model.best_score['train']['gan_eval']:.6f}")
    return model



In [10]:
model = Autoencoder(input_dim=INPUT_DIM, encoder_layer_sizes=LAYERS)
ds_train = TabularDataset(X_train)
dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True)
model = train_autoencoder(model, dl_train, num_epochs=NUM_EPOCHS, lr=LR, device=device)


Epoch 1/50, loss = 0.474965
Epoch 10/50, loss = 0.250419
Epoch 20/50, loss = 0.207601
Epoch 30/50, loss = 0.173783
Epoch 40/50, loss = 0.166252
Epoch 50/50, loss = 0.155752


In [17]:
y_train = (y_train == "BAJA+2").astype(int)
y_test = (y_test == "BAJA+2").astype(int)


In [27]:
X_train_embeddings = get_embeddings(model, X_train, device=device)
X_test_embeddings = get_embeddings(model, X_test, device=device)

#lgbm_model = train_lgbm(X_train_embeddings, y_train, X_test_embeddings, y_test, w_train, w_test)


In [ ]:
X_train_embeddings = get_embeddings(model, X_train, device=device)
X_test_embeddings = get_embeddings(model, X_test, device=device)


X_train_embeddings = np.concatenate([X_train_embeddings, X_train], axis=1)
X_test_embeddings = np.concatenate([X_test_embeddings, X_test], axis=1)




In [23]:
lgbm_model = train_lgbm(X_train_embeddings, y_train, X_test_embeddings, y_test, w_train, w_test)

Training until validation scores don't improve for 200 rounds
[100]	train's gan_eval: 5.6324e+08	test's gan_eval: 3.2974e+08
[200]	train's gan_eval: 5.9596e+08	test's gan_eval: 3.31e+08
[300]	train's gan_eval: 6.1822e+08	test's gan_eval: 3.3052e+08
[400]	train's gan_eval: 6.4046e+08	test's gan_eval: 3.32e+08
[500]	train's gan_eval: 6.6388e+08	test's gan_eval: 3.352e+08
[600]	train's gan_eval: 6.8372e+08	test's gan_eval: 3.3644e+08
[700]	train's gan_eval: 6.9834e+08	test's gan_eval: 3.3692e+08
[800]	train's gan_eval: 7.1562e+08	test's gan_eval: 3.3942e+08
[900]	train's gan_eval: 7.3188e+08	test's gan_eval: 3.3984e+08
[1000]	train's gan_eval: 7.4496e+08	test's gan_eval: 3.42e+08
[1100]	train's gan_eval: 7.6096e+08	test's gan_eval: 3.416e+08
[1200]	train's gan_eval: 7.7438e+08	test's gan_eval: 3.4168e+08
Early stopping, best iteration is:
[1049]	train's gan_eval: 7.5518e+08	test's gan_eval: 3.4224e+08
✅ Modelo entrenado exitosamente!
Mejor iteración: 1049
Mejor ganancia de validación: 342

In [24]:
lgbm_model = train_lgbm(X_train, y_train, X_test, y_test, w_train, w_test)

Training until validation scores don't improve for 200 rounds
[100]	train's gan_eval: 5.5464e+08	test's gan_eval: 3.2352e+08
[200]	train's gan_eval: 5.824e+08	test's gan_eval: 3.2664e+08
[300]	train's gan_eval: 6.1032e+08	test's gan_eval: 3.3038e+08
[400]	train's gan_eval: 6.339e+08	test's gan_eval: 3.3294e+08
[500]	train's gan_eval: 6.5202e+08	test's gan_eval: 3.3488e+08
[600]	train's gan_eval: 6.7132e+08	test's gan_eval: 3.3642e+08
[700]	train's gan_eval: 6.8402e+08	test's gan_eval: 3.385e+08
[800]	train's gan_eval: 6.976e+08	test's gan_eval: 3.397e+08
[900]	train's gan_eval: 7.1144e+08	test's gan_eval: 3.4032e+08
[1000]	train's gan_eval: 7.2684e+08	test's gan_eval: 3.394e+08
Early stopping, best iteration is:
[861]	train's gan_eval: 7.0628e+08	test's gan_eval: 3.4182e+08
✅ Modelo entrenado exitosamente!
Mejor iteración: 861
Mejor ganancia de validación: 341820000.000000
Mejor ganancia de training: 706280000.000000


In [25]:
import optuna

/Users/frjofre/Documents/EyF/eyf/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def objective(trial):

    num_leaves = trial.suggest_int('num_leaves', 8, 100),
    learning_rate = trial.suggest_float('learning_rate', 0.005, 0.3), # mas bajo, más iteraciones necesita
    min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 5, 10000),
    feature_fraction = trial.suggest_float('feature_fraction', 0.1, 1.0),
    bagging_fraction = trial.suggest_float('bagging_fraction', 0.1, 1.0),

    params = {
        'objective': 'binary',
        'metric': 'custom',
        'boosting_type': 'gbdt',
        'first_metric_only': True,
        'boost_from_average': True,
        'feature_pre_filter': False,
        'max_bin': 31,
        'num_leaves': num_leaves,
        'learning_rate': learning_rate,
        'min_data_in_leaf': min_data_in_leaf,
        'feature_fraction': feature_fraction,
        'bagging_fraction': bagging_fraction,
        'seed': 42,
        'verbose': -1,
        'num_threads':10,
        'early_stopping_rounds': int(75)

    }
    train_data = lgb.Dataset(X_train_embeddings,
                              label=y_train, # eligir la clase
                              weight=w_train)
    cv_results = lgb.cv(
        params,
        train_data,
        num_boost_round=2000, # modificar, subit y subir... y descomentar la línea inferior,
        feval=lgb_gan_eval,
        stratified=True,
        nfold=5,
        seed=42,
    )
    max_gan = max(cv_results['valid gan_eval-mean'])
    best_iter = cv_results['valid gan_eval-mean'].index(max_gan) + 1

    # Guardamos cual es la mejor iteración del modelo
    trial.set_user_attr("best_iter", best_iter)

    return max_gan * 5

# Crear estudio sin storage persistente (se guardará en CSV al final)
study_name = "exp_01_AE_lgbm"
csv_path = DATA_PATH + f"{study_name}.csv"

study = optuna.create_study(
    direction="maximize",
    study_name=study_name
)

[I 2025-10-02 00:37:08,099] A new study created in memory with name: exp_01_AE_lgbm


In [33]:
print("Iniciando optimización de hiperparámetros...")
study.optimize(objective, n_trials=1500)  # Ajusta el número de trials según necesites

print("Optimización completada!")
print(f"Mejor valor: {study.best_value}")
print(f"Mejores parámetros: {study.best_params}")

Iniciando optimización de hiperparámetros...


[I 2025-10-02 00:37:21,530] Trial 0 finished with value: 341740000.0 and parameters: {'num_leaves': 53, 'learning_rate': 0.10447585447235058, 'min_data_in_leaf': 5949, 'feature_fraction': 0.19721640666874019, 'bagging_fraction': 0.4456585583714312}. Best is trial 0 with value: 341740000.0.
[I 2025-10-02 00:37:31,227] Trial 1 finished with value: 360480000.0 and parameters: {'num_leaves': 56, 'learning_rate': 0.10162490126927234, 'min_data_in_leaf': 5068, 'feature_fraction': 0.8361547064476299, 'bagging_fraction': 0.24762807887280802}. Best is trial 1 with value: 360480000.0.
[I 2025-10-02 00:37:53,944] Trial 2 finished with value: 355940000.0 and parameters: {'num_leaves': 100, 'learning_rate': 0.025085942692190225, 'min_data_in_leaf': 1240, 'feature_fraction': 0.6466797466491229, 'bagging_fraction': 0.6165863239146173}. Best is trial 1 with value: 360480000.0.
[I 2025-10-02 00:38:09,800] Trial 3 finished with value: 368340000.0 and parameters: {'num_leaves': 47, 'learning_rate': 0.038

Optimización completada!
Mejor valor: 379420000.0
Mejores parámetros: {'num_leaves': 23, 'learning_rate': 0.01557589250807279, 'min_data_in_leaf': 5863, 'feature_fraction': 0.5703675405264673, 'bagging_fraction': 0.8181603263828602}


In [ ]:
study_name = "exp_01_AE_lgbm"

csv_path = "../studies/" + f"{study_name}.csv"
# Guardar resultados en CSV
print(f"Guardando resultados en: {csv_path}")

# Crear DataFrame con todos los trials
trials_data = []
for trial in study.trials:
    trial_data = {
        'trial_number': trial.number,
        'value': trial.value,
        'state': trial.state.name,
        'datetime_start': trial.datetime_start,
        'datetime_complete': trial.datetime_complete,
        'duration': (trial.datetime_complete - trial.datetime_start).total_seconds() if trial.datetime_complete and trial.datetime_start else None
    }
    
    # Agregar parámetros
    for param_name, param_value in trial.params.items():
        trial_data[f'param_{param_name}'] = param_value
    
    # Agregar user attributes (como best_iter)
    for attr_name, attr_value in trial.user_attrs.items():
        trial_data[f'user_attr_{attr_name}'] = attr_value
    
    trials_data.append(trial_data)

# Convertir a DataFrame y guardar
df_results = pd.DataFrame(trials_data)
df_results.to_csv(csv_path, index=False)

print(f"✅ Resultados guardados exitosamente en {csv_path}")
print(f"Total de trials: {len(df_results)}")

# Mostrar un resumen
print("\n=== RESUMEN DE OPTIMIZACIÓN ===")
print(f"Mejor trial: {study.best_trial.number}")
print(f"Mejor valor: {study.best_value:.6f}")
print("Mejores parámetros:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")
print("================================")


Guardando resultados en: ../dataexp_01_AE_lgbm.csv
✅ Resultados guardados exitosamente en ../dataexp_01_AE_lgbm.csv
Total de trials: 1500

=== RESUMEN DE OPTIMIZACIÓN ===
Mejor trial: 1327
Mejor valor: 379420000.000000
Mejores parámetros:
  num_leaves: 23
  learning_rate: 0.01557589250807279
  min_data_in_leaf: 5863
  feature_fraction: 0.5703675405264673
  bagging_fraction: 0.8181603263828602
